# Block 3 · Trees, ensembles & honest evaluation
> Data Mining · SGH Warsaw School of Economics (SMMD-ADA / SMMD-AAB)
> Course anchor dataset: retail-credit / PD portfolio

**Business framing.** Same lender, same portfolio. Today we let models *bend* — trees, forests, gradient boosting — and, more importantly, we upgrade how we judge them: cross-validation, ROC/AUC, calibration, and a threshold with a price tag. Along the way we detonate the leakage trap that has been sitting in the data since Block 1.

**Learning goals for this lab**
1. Cross-validate instead of trusting one split; report mean ± spread.
2. Grow, read, and leash a decision tree.
3. Run the tournament: tree vs random forest vs GBDT vs the logistic baseline.
4. Judge the winner with ROC/AUC, PR, and a calibration curve.
5. Reproduce the leakage detonation — and recognise the smell forever.
6. Choose a threshold with an expected-cost curve.

**How to work.** Run cells top to bottom. Before you trust any result: **Kernel → Restart & Run All.**

In [ ]:
import sys
sys.path.append("../lecture_1")   # shared course data loader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.max_columns', 30)

from finance_data import load_taiwan, taiwan_features
df = load_taiwan()

# Block-2 feature matrix (honest features only!)
X = taiwan_features(df)
y = df["default"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

## 1. Cross-validation: a distribution, not a lucky draw
One test split is one draw. Cross-validation rotates the held-out fold so every training row gets scored once by a model that never saw it — and gives us a **spread**, not just a point.

Note the hygiene: models with preprocessing go into CV **as pipelines**, so the scaler is refit inside every fold.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(5, shuffle=True, random_state=42)
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

scores = cross_val_score(baseline, X_train, y_train, cv=cv, scoring="roc_auc")
print("fold AUCs:", scores.round(3))
print(f"CV AUC: {scores.mean():.3f} ± {scores.std():.3f}")

Remember that spread (~±0.02). Any model that "wins" by less has not won anything yet.

## 2. One tree: grow it, read it, leash it

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(max_depth=2, min_samples_leaf=50, random_state=42)
tree.fit(X_train, y_train)

plt.figure(figsize=(14, 5))
plot_tree(tree, feature_names=list(X.columns), class_names=["repaid", "default"],
          filled=True, impurity=False, proportion=True, rounded=True, fontsize=10)
plt.show()

Read the root split aloud. The tree opens on `pay_delay_recent` at the **two-months-behind cliff** — the same signal the logistic model crowned in Block 2. Two very different algorithms, one verdict: payment history is king.

Now the leash. Watch the U-curve appear as we free the tree:

In [ ]:
from sklearn.metrics import roc_auc_score

rows = []
for depth in range(1, 15):
    t = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    rows.append({
        "depth": depth,
        "train AUC": roc_auc_score(y_train, t.predict_proba(X_train)[:, 1]),
        "test AUC":  roc_auc_score(y_test,  t.predict_proba(X_test)[:, 1]),
    })
curve = pd.DataFrame(rows).set_index("depth")

curve.plot(figsize=(7, 3.2), marker="o", ms=3, color=["#2E6DA4", "#C83C28"])
plt.ylabel("AUC"); plt.xlabel("max_depth (the leash)")
plt.tight_layout(); plt.show()
curve.round(3).T

## 3. The tournament
Four models, identical CV protocol. Trees don't need the scaler; the logistic pipeline keeps it.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
    "logistic (baseline)": baseline,
    "tree (depth 4)": DecisionTreeClassifier(max_depth=4, min_samples_leaf=50,
                                             random_state=42),
    "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=10,
                                            random_state=42, n_jobs=-1),
    "GBDT (tuned)": HistGradientBoostingClassifier(
        learning_rate=0.03, max_leaf_nodes=7, min_samples_leaf=60,
        early_stopping=True, random_state=42),
}

results = {}
for name, model in models.items():
    s = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    results[name] = s
    print(f"{name:22s} CV AUC {s.mean():.3f} ± {s.std():.3f}")

> **Try it yourself:** replace the tuned GBDT with `HistGradientBoostingClassifier(random_state=42)` (pure defaults) and re-run. On 21,000 training rows the defaults are already excellent — the same defaults on Block 1's small synthetic portfolio overfit badly. Leashes are sized to the data.

**Every rung of the ladder pays** here: logistic → tree → forest/GBDT gains ~4 AUC points end to end, and the spreads (±0.01) certify the gaps as real. This is what complexity *earning its keep* looks like on real behavioural data — the repayment-status cliff and delay×limit interactions are exactly what linear models must be hand-fed.

Final confirmation on the untouched test set:

In [ ]:
test_auc = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    test_auc[name] = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
pd.Series(test_auc, name="test AUC").round(3)

## 4. Judging the ranking: ROC/AUC, Gini, PR

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

proba_lo = models["logistic (baseline)"].predict_proba(X_test)[:, 1]
proba_gb = models["GBDT (tuned)"].predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, proba, color in [("logistic", proba_lo, "#2E6DA4"),
                           ("GBDT", proba_gb, "#E8A33D")]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, color=color, label=f"{name} AUC {auc:.3f} (Gini {2*auc-1:.2f})")
    prec, rec, _ = precision_recall_curve(y_test, proba)
    axes[1].plot(rec, prec, color=color,
                 label=f"{name} AP {average_precision_score(y_test, proba):.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].legend()
axes[1].axhline(y_test.mean(), ls="--", color="gray")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].legend()
plt.tight_layout(); plt.show()

**Credit dialect:** risk managers say **Gini = 2·AUC − 1** (logistic ≈ 0.46, GBDT ≈ 0.55 — normal furniture for retail PD) — not to be confused with the Gini *impurity* inside trees. Same statistician, different formula.

## 5. Calibration: is a predicted 0.3 really a 0.3?
Ranking pays for approval order; **calibration** pays for pricing. Bin customers by predicted PD and compare with the observed default rate per bin.

In [ ]:
from sklearn.calibration import calibration_curve

plt.figure(figsize=(5.5, 4))
for name, proba, color, marker in [("logistic", proba_lo, "#2E6DA4", "o"),
                                   ("GBDT", proba_gb, "#E8A33D", "s")]:
    frac, mean_pred = calibration_curve(y_test, proba, n_bins=10, strategy="quantile")
    plt.plot(mean_pred, frac, marker=marker, ms=5, color=color, label=name)
plt.plot([0, 0.75], [0, 0.75], "--", color="gray", label="perfect")
plt.xlabel("mean predicted PD (bin)"); plt.ylabel("observed default rate (bin)")
plt.legend(); plt.tight_layout(); plt.show()

If a model rides above the diagonal it under-predicts risk; below, it over-predicts. Repairs: `sklearn.calibration.CalibratedClassifierCV` (Platt / isotonic) — exercise 4.

## 6. 💣 The detonation
`collections_contact` has been in the dataframe since Block 1, flagged *do not use*: collections calls customers mostly **after** they default. Let's do the forbidden thing, with instruments attached.

In [ ]:
X_leak = X.assign(collections_contact=df["collections_contact"])
Xl_train, Xl_test = X_leak.loc[X_train.index], X_leak.loc[X_test.index]

leaky = HistGradientBoostingClassifier(learning_rate=0.03, max_leaf_nodes=7,
                                       min_samples_leaf=60, early_stopping=True,
                                       random_state=42)
leaky.fit(Xl_train, y_train)
auc_leaky = roc_auc_score(y_test, leaky.predict_proba(Xl_test)[:, 1])

print(f"honest features:          test AUC {test_auc['GBDT (tuned)']:.3f}")
print(f"+ collections_contact:    test AUC {auc_leaky:.3f}")
print("Best model of the course. Completely worthless.")

In [ ]:
# The leak even LOOKS like the best feature — importance crowns it instantly:
from sklearn.inspection import permutation_importance
perm = permutation_importance(leaky, Xl_test, y_test, n_repeats=10,
                              random_state=42, scoring="roc_auc", n_jobs=-1)
pd.Series(perm.importances_mean, index=X_leak.columns)\
  .sort_values(ascending=False).round(4).head(5)

Notice what did **not** happen: no error, no warning. CV would have loved this feature too, because the leak sits on both sides of every split. The only working alarms are *yours*: the time-travel test, and "too good to be true". Behavioural models live around Gini 0.5–0.7; a Gini of 0.91 is not a triumph, it is a subpoena.

## 7. From metrics to money: choosing the threshold
Price the confusion-matrix cells and sweep the threshold. Here a missed defaulter costs 3× a falsely declined client (deliberately conservative for our 22% default rate; real low-default books push 10–50×).

In [ ]:
C_FN, C_FP = 3.0, 1.0
thresholds = np.linspace(0.02, 0.9, 120)
costs = []
for t in thresholds:
    pred = proba_gb >= t
    fn = ((pred == 0) & (y_test == 1)).sum()
    fp = ((pred == 1) & (y_test == 0)).sum()
    costs.append((fn * C_FN + fp * C_FP) / len(y_test))
costs = np.array(costs)
best = thresholds[costs.argmin()]

plt.figure(figsize=(7, 3.2))
plt.plot(thresholds, costs, color="#2E6DA4")
plt.axvline(best, color="#C83C28", ls=":")
plt.axvline(0.5, color="gray", ls="--")
plt.xlabel("threshold"); plt.ylabel("expected cost / customer")
plt.tight_layout(); plt.show()

cost_05 = costs[np.abs(thresholds - 0.5).argmin()]
print(f"optimal threshold {best:.2f}: cost {costs.min():.3f}")
print(f"default 0.5:               cost {cost_05:.3f}")
print(f"savings from moving the threshold: {1 - costs.min()/cost_05:.0%}")

Same model, better decision: moving the cut-off from 0.5 to ~0.26 cuts expected cost by ~11%. **The threshold is a business decision expressed in code** — and now you can price it.

## Optional extra: feed raw NaNs to the GBDT
Histogram-based GBDT treats "missing" as just another bin — no imputation needed. Let's knock out 20% of `limit_bal` at random and watch the model shrug. (Compare Block 1, where we had to *choose* an imputation strategy for the linear world.)

In [ ]:
# knock out 20% of limit_bal, refit the same GBDT, compare test AUC
rng = np.random.default_rng(42)
X_nan = X.copy()
mask = rng.random(len(X_nan)) < 0.20
X_nan.loc[mask, "limit_bal"] = np.nan

Xn_train, Xn_test = X_nan.loc[X_train.index], X_nan.loc[X_test.index]
gb_nan = HistGradientBoostingClassifier(learning_rate=0.03, max_leaf_nodes=7,
                                        min_samples_leaf=60, early_stopping=True,
                                        random_state=42).fit(Xn_train, y_train)
print(f"complete data:      test AUC {test_auc['GBDT (tuned)']:.3f}")
print(f"20% NaN limit_bal:  test AUC {roc_auc_score(y_test, gb_nan.predict_proba(Xn_test)[:, 1]):.3f}")

## 8. Exercises
1. Add the six raw bill amounts (`bill_amt_apr` … `bill_amt_sep`) to the feature matrix and re-run the tournament. Does any model's CV score move by more than the fold spread — and which model family benefits most from the extra raw columns?
2. **SVM entry:** fit `make_pipeline(StandardScaler(), SVC(kernel="rbf"))` on a 6,000-row subsample (mind the n² cost) and score AUC via `decision_function`. Where does it land in the tournament, and why might kernels lag trees here?
3. Give the forest a proper leash sweep: `min_samples_leaf` ∈ {2, 5, 10, 25, 50} under CV. Where is its U-curve?
4. Calibrate the GBDT with `CalibratedClassifierCV(method="isotonic")` and redraw the reliability diagram. What changed, and did AUC move? (Should it?)
5. Re-run the expected-cost analysis with C_FN/C_FP = 10. Where does the optimal threshold go, and what does the model *do* at that threshold? Explain to a risk manager in two sentences.
6. **Monotonic constraint (optional):** refit the GBDT with `monotonic_cst` forcing risk non-decreasing in `utilisation`. Compare AUCs and argue why a regulator might prefer the constrained model even if it scores marginally lower.

*Hand-in for the project team: your tournament table (CV mean ± sd per model), the winner's ROC and calibration plots, and your cost-based threshold with a two-sentence justification.*

**"Done" for today** = you can say which model won, whether the gap beats the CV spread, and at which threshold you would run it — with a cost curve to back you up.

In [ ]:
# scratch cell for the exercises